In [ ]:
from IPython.display import display
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
X_train = pd.read_pickle('X_train.pkl')
X_test = pd.read_pickle('X_test.pkl')
y_train = pd.read_pickle('y_train.pkl')
y_test = pd.read_pickle('y_test.pkl')

In [ ]:
# przygotowanie danych do klastrowania
zmienne_do_klastrowania = ['mileage', 'engine_displacement', 'power', 'car_age', 'vehicle_brand', 'vehicle_model']

X_train_clus = X_train[zmienne_do_klastrowania].copy()
X_train_clus['price'] = y_train

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clus)

# metoda łokcia do wyboru liczby klastrów
inertia = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_train_scaled)
    inertia.append(kmeans.inertia_)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o', linestyle='--', color='teal', linewidth=2)
plt.xlabel('Liczba klastrów (k)', fontsize=12)
plt.ylabel('Inercja', fontsize=12)
plt.tight_layout()
plt.show()

# końcowa liczba klastrów wybrana na podstawie wykresu łokcia
WYBRANA_LICZBA_KLASTRÓW = 3
kmeans_final = KMeans(n_clusters=WYBRANA_LICZBA_KLASTRÓW, random_state=42, n_init=10)

# Przypisanie dla Train
X_train['cluster_id'] = kmeans_final.fit_predict(X_train_scaled)

# Przypisanie dla Test
X_test_clus = X_test[zmienne_do_klastrowania].copy()
X_test_clus['price'] = y_test
X_test_scaled = scaler.transform(X_test_clus)
X_test['cluster_id'] = kmeans_final.predict(X_test_scaled)

In [ ]:
# boxplot dla ceny w zależności od klastra
df_analiza = X_train[zmienne_do_klastrowania].copy()
df_analiza['price'] = y_train
df_analiza['cluster_id'] = X_train['cluster_id']

# Przygotowanie tabeli
profile_klastrow = df_analiza.groupby('cluster_id').mean().round(1)
profile_klastrow['count'] = df_analiza['cluster_id'].value_counts()


tlumaczenia_tabeli = {
    'mileage': 'Przebieg (km)',
    'engine_displacement': 'Pojemność silnika (cm³)',
    'power': 'Moc silnika (KM)',
    'car_age': 'Wiek pojazdu (lata)',
    'vehicle_brand': 'Wartość marki (PLN)',
    'vehicle_model': 'Wartość modelu (PLN)',
    'price': 'Średnia cena (PLN)',
    'count': 'Liczba pojazdów'
}

profile_klastrow = profile_klastrow.rename(columns=tlumaczenia_tabeli)
profile_klastrow.index.name = 'Klaster'

print("--- PROFILE CHARAKTERYSTYKI KLASTRÓW ---")
display(profile_klastrow)

sns.set_theme(style="ticks")

plt.figure(figsize=(10, 6))

df_analiza['nazwa_klastra'] = df_analiza['cluster_id'].apply(lambda x: f"Klaster {x}")
kolejnosc_klastrow = sorted(df_analiza['nazwa_klastra'].unique())

# Wykres ramkowy
ax = sns.boxplot(
    data=df_analiza, 
    x='nazwa_klastra', 
    y='price', 
    hue='nazwa_klastra',
    order=kolejnosc_klastrow,
    palette='muted', 
    legend=False,
    flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.15, 'color': '#4c72b0'}
)

plt.ylabel("Cena ofertowa (PLN)", fontsize=11)
plt.xlabel("") 

def format_duzych_liczb(x, pos):
    return "{:,}".format(int(x)).replace(',', ' ')
    
ax.yaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))

ax.grid(True, linestyle='--', alpha=0.5, axis='y')
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# eksportujemy profile klastrów do pliku plk
# zapis całych danych do późniejszego XAI
X_train.to_pickle('X_train_with_clusters.pkl')
X_test.to_pickle('X_test_with_clusters.pkl')

joblib.dump(scaler, 'k_means_scaler.pkl')
joblib.dump(kmeans_final, 'k_means_model.pkl')
print("Dane z klastrami oraz obiekty K-Means zostały zapisane na dysku!")